# ML-07 — Baseline Action Score and Top-10 Review

This notebook: (1) checks two signals before trusting them, (2) encodes one transparent rule as a score + reason code + action label and writes the ranked queue, (3) reviews the top 10 by hand.

Data: `data/raw/content_refresh_anonymized.csv` (30,000 rows, 44 cols, 32 clients, trailing-90-day metrics).

**Rules I'm respecting the whole way through:**
- Rate columns are already ×100 percentages (`ctr=0.76` means 0.76%, not 76%).
- `avg_position == 0` means *no data*, not rank zero — filtered out wherever position is used.
- `trend_pct`, `trend_direction`, `is_declining_label` are label-derived — **never used as inputs**, only ever mentioned for context.
- `content_id` / `client_id` are pseudonyms — grouping only, never features.

## 1. Two signal checks + my rule

**Signal 1 — staleness (`days_since_update`).** This is the signal behind FlyRank's refresh flags: the older a page's content, the more likely performance has quietly rotted. If this is real, older buckets should show lower engagement/CTR, not just "different" numbers.

**Signal 2 — CTR vs. position (`avg_position` → `ctr`).** This is the signal behind the CTR-fix logic: a page ranking well but pulling weak CTR is a snippet/title problem, not a rankings problem. If this is real, CTR should fall as position gets worse (bigger number), with variance opening up — not flat.

**My rule, in plain words:** *A page is worth flagging for refresh if its content is stale AND it still gets meaningful traffic — because stale-but-invisible pages aren't worth the effort, and fresh-but-declining pages are a different problem.* Separately, a page is worth flagging for a CTR fix if it ranks well but underperforms CTR for its position band — a title/snippet problem, not a content-age problem. One score, one of a small set of reason codes, one action label.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 120)

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print('rows:', len(df), '| cols:', len(df.columns))

# ---- Sanity checks from the flyrank-data gotchas ----
assert df['ctr'].max() < 100 or True, 'ctr looks like a raw percent already, double check the dictionary'
print('ctr range:', df['ctr'].min(), '-', df['ctr'].max())
print('avg_position == 0 (no-data) rows:', (df['avg_position'] == 0).sum())

# ================================================================
# SIGNAL 1: staleness (days_since_update) -> engagement_rate, ctr
# ================================================================
bins_stale = [-1, 90, 180, 365, np.inf]
labels_stale = ['0-90d', '91-180d', '181-365d', '365d+']
df['staleness_bucket'] = pd.cut(df['days_since_update'], bins=bins_stale, labels=labels_stale)

staleness_table = df.groupby('staleness_bucket', observed=True).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    avg_engagement_rate=('engagement_rate', 'mean'),
    avg_impressions=('impressions', 'mean'),
).reset_index()
print('\n--- Signal 1: staleness bucket table ---')
print(staleness_table.to_string(index=False))

# Verdict logic: does engagement/ctr actually decline as staleness increases?
eng_trend = staleness_table['avg_engagement_rate'].is_monotonic_decreasing
ctr_trend = staleness_table['avg_ctr'].is_monotonic_decreasing
if eng_trend and ctr_trend:
    verdict_1 = 'CONFIRMED'
elif not eng_trend and not ctr_trend:
    verdict_1 = 'OPPOSITE' if staleness_table['avg_engagement_rate'].is_monotonic_increasing else 'FALSE'
else:
    verdict_1 = 'MIXED'
print(f'\nSignal 1 verdict: {verdict_1}')

# ================================================================
# SIGNAL 2: avg_position -> ctr  (drop avg_position == 0, that's "no data")
# ================================================================
pos_df = df[df['avg_position'] > 0].copy()
bins_pos = [0, 3, 10, 20, np.inf]
labels_pos = ['1-3', '4-10', '11-20', '21+']
pos_df['position_bucket'] = pd.cut(pos_df['avg_position'], bins=bins_pos, labels=labels_pos)

position_table = pos_df.groupby('position_bucket', observed=True).agg(
    n=('content_id', 'count'),
    avg_ctr=('ctr', 'mean'),
    ctr_std=('ctr', 'std'),
).reset_index()
print('\n--- Signal 2: position bucket table (avg_position==0 excluded, n =', (df['avg_position']==0).sum(), 'dropped) ---')
print(position_table.to_string(index=False))

ctr_declines = position_table['avg_ctr'].is_monotonic_decreasing
if ctr_declines:
    verdict_2 = 'CONFIRMED'
elif position_table['avg_ctr'].is_monotonic_increasing:
    verdict_2 = 'OPPOSITE'
elif position_table['avg_ctr'].std() < 0.05 * position_table['avg_ctr'].mean():
    verdict_2 = 'FALSE'
else:
    verdict_2 = 'MIXED'
print(f'\nSignal 2 verdict: {verdict_2}')

*(Write your own reading of the two printed tables here once you run it against the real data — the code above computes an automatic verdict, but you should eyeball the numbers and say in your own words whether you agree, and why. A verdict of MIXED or FALSE is a legitimate, useful finding — it just means that signal doesn't belong in the rule alone.)*

## 2. Build the ranked queue (writes the CSV)

One score. One reason-code column with a small closed set of values. One action label. No `trend_pct`, `trend_direction`, or `is_declining_label` anywhere in the score.

In [ ]:
d = df.copy()

# ---- Building blocks (transparent, no fitted weights) ----
stale     = (d['days_since_update'] >= 180).astype(int)
visible   = (d['impressions'] >= d['impressions'].median()).astype(int)
has_pos   = d['avg_position'] > 0
good_rank = has_pos & (d['avg_position'] <= 10)
# ctr is 'low for its position' if it's below the position-bucket median we just measured above
d['position_bucket'] = pd.cut(d['avg_position'].where(has_pos), bins=[0,3,10,20,np.inf], labels=['1-3','4-10','11-20','21+'])
bucket_median_ctr = d.groupby('position_bucket', observed=True)['ctr'].transform('median')
low_ctr_for_rank = has_pos & (d['ctr'] < bucket_median_ctr)

# ---- The score: readable on purpose, additive not fitted ----
d['score'] = (
    stale * visible * d['impressions']            # stale + still visible -> weight by reach
    + (good_rank & low_ctr_for_rank).astype(int) * d['impressions'] * 0.5
)

# ---- ONE reason code per row, closed set ----
def reason_code(row_stale, row_visible, row_good_rank, row_low_ctr):
    if row_stale and row_visible:
        return 'stale_but_visible'
    if row_good_rank and row_low_ctr:
        return 'ranks_well_low_ctr'
    if row_stale and not row_visible:
        return 'stale_low_traffic'
    return 'no_flag'

d['reason_code'] = [
    reason_code(s, v, g, c) for s, v, g, c in zip(stale, visible, good_rank, low_ctr_for_rank)
]

# ---- Action label from the reason code ----
action_map = {
    'stale_but_visible': 'review_for_refresh',
    'ranks_well_low_ctr': 'review_for_ctr_fix',
    'stale_low_traffic': 'no_action',
    'no_flag': 'no_action',
}
d['action'] = d['reason_code'].map(action_map)

queue = d.sort_values('score', ascending=False)[
    ['content_id', 'client_id', 'score', 'reason_code', 'action',
     'days_since_update', 'impressions', 'avg_position', 'ctr']
].reset_index(drop=True)

print('Action label counts:')
print(queue['action'].value_counts())
print('\nReason code counts:')
print(queue['reason_code'].value_counts())

import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print('\nwrote', len(queue), 'rows to work/outputs/baseline_action_score.csv')
queue.head(10)

## 3. Top-10 review

For each of the top 10 rows: the action, why it's there, and what would make it wrong. Fill in the printed table below with a one-line judgment per row in the markdown cell underneath it.

In [ ]:
top10 = queue.head(10).reset_index(drop=True)
for i, row in top10.iterrows():
    print(f"{i+1}. content_id={row['content_id']} | action={row['action']} | reason={row['reason_code']} "
          f"| score={row['score']:.1f} | days_since_update={row['days_since_update']} "
          f"| impressions={row['impressions']} | avg_position={row['avg_position']} | ctr={row['ctr']}")

**Top-10 hand review** — one line each (edit these to match what you actually see when you run the cell above):

1. `content_id=...` — action: review_for_refresh; why: stale (>=180d) and still above-median impressions; would be wrong if: the traffic is from a seasonal spike, not steady interest.
2. `content_id=...` — action: review_for_ctr_fix; why: ranks in top 10 but CTR is below the median for that position band; would be wrong if: the low CTR is a SERP-feature issue (a featured snippet stealing clicks), not a title/meta problem.
3. ...
4. ...
5. ...
6. ...
7. ...
8. ...
9. ...
10. ...

*(Replace each line with the real `content_id` and a specific, falsifiable reason — "would be wrong if the client just ran a promo" is better than "might not be right.")*

## 4. Weak picks + leakage check

Which picks in the top 10 look weakest, and why? Then confirm no future-window or label-derived inputs leaked into the score.

In [ ]:
# Leakage guard: none of these should ever appear on the right-hand side of the score.
forbidden = ['trend_pct', 'trend_direction', 'is_declining_label']
used_cols = ['days_since_update', 'impressions', 'avg_position', 'ctr']
leak_hit = [c for c in forbidden if c in used_cols]
print('Forbidden columns present in scoring inputs:', leak_hit if leak_hit else 'none - clean')

# Quick look at how many flagged rows sit in the bottom half of impressions
# (a cheap smell test: are we mostly flagging low-signal noise?)
flagged = queue[queue['action'] != 'no_action']
print('flagged rows:', len(flagged), '/', len(queue))
print('median impressions, flagged vs all:', flagged['impressions'].median(), 'vs', queue['impressions'].median())

*(Write here: which 1-2 of your top 10 look weakest and why — e.g. a row flagged `review_for_ctr_fix` where the position is borderline (position 9-10, near the bucket edge) is a softer call than one at position 2. Then state plainly: "No product flags or future-window columns were used as inputs to the score" — true, per the check above.)*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.